# 一次性取数：把 BF16 base 挂成 Kaggle 只读 Input

这个 notebook **只做一件事**：把 `unsloth/Qwen3-8B-GGUF` 里 pinned revision 的
`Qwen3-8B-BF16.gguf`（16,388,044,384 B，sha256 `5e416a2020fe63e76ea13c8979be35fc6070aaf3578f7876400c55c2f5c3eb30`）
下载到 `/kaggle/working`，校验字节数与 sha256，然后你 `Save Version` 把它变成 Output。

主 notebook（`trlab_kaggle.ipynb`）的 S4 阶段会通过 `Add Input` 用它当量化 base。
**挂一次，之后每个 Session 都零点击、不再下载。**

## 操作步骤（共 6 次点击）

1. Kaggle → `File` → `Import Notebook` → 上传本文件
2. 右侧 `Settings` → `Internet` = **On**
3. `Run` → `Run all`（约 15–30 分钟，取决于 Kaggle 出口带宽；最后一步 sha256 校验约 1–2 分钟）
4. `Save Version`（**Quick Save 即可**，不要 Save & Run All —— 后者会从头重跑）
5. 回到主 notebook → 右侧 `Add Input` → 选本 notebook 的输出
6. 主 notebook 里把 `RUN_S4 = True` → `Run All`

## 为什么需要这一步

`/kaggle/working` 只有 20 GB，S2 编译后实测只剩约 19.07 GiB，而
BF16（15.26 GiB）+ IQ2_XS（2.36 GiB）+ IQ4_XS（4.34 GiB）= 21.96 GiB，装不下。
只读 Input 不占 working 配额，是唯一能一次 Run All 跑完 S4+S5 的办法。

> 如果 Kaggle 给这个会话提供了 working 之外的暂存盘，主 notebook 会自己探测到并
> 直接把 BF16 下到那里 —— 那种情况下本 notebook 可以整个跳过。


In [ ]:
# -*- coding: utf-8 -*-
"""下载 pinned BF16 并强校验。只用 Kaggle 镜像自带的东西，不 pip install。"""
import hashlib
import os
import time
from pathlib import Path

REPO_ID     = "unsloth/Qwen3-8B-GGUF"
REVISION    = "a6adef130ffb23ddaf1a62fec9dced968c9bc482"   # 与主 notebook 的 S4_PINNED 一致
FILENAME    = "Qwen3-8B-BF16.gguf"
EXPECT_BYTES  = 16388044384
EXPECT_SHA256 = "5e416a2020fe63e76ea13c8979be35fc6070aaf3578f7876400c55c2f5c3eb30"

DST = Path("/kaggle/working") / FILENAME
PART = DST.with_name(DST.name + ".partial")

t0 = time.time()


def human(n):
    for unit in ("B", "KiB", "MiB", "GiB"):
        if n < 1024 or unit == "GiB":
            return f"{n:.2f} {unit}"
        n /= 1024


def sha256_of(path, chunk=8 << 20):
    h = hashlib.sha256()
    done = 0
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
            done += len(b)
            if done % (1 << 30) < chunk:
                print(f"  ... 已校验 {human(done)}（{time.time() - t0:.0f}s）", flush=True)
    return h.hexdigest()


if DST.exists() and DST.stat().st_size == EXPECT_BYTES:
    print(f"已存在且字节数正确：{DST}（{human(EXPECT_BYTES)}）→ 跳过下载，只校验 sha256")
else:
    # huggingface_hub 在 Kaggle 镜像里通常有；没有就退回标准库 urllib
    try:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, revision=REVISION,
                            local_dir="/kaggle/working")
        print(f"hf_hub_download → {p}（{human(os.path.getsize(p))}，{time.time() - t0:.0f}s）")
        if os.path.abspath(p) != os.path.abspath(DST):
            os.replace(p, DST)
    except ImportError:
        url = (f"https://huggingface.co/{REPO_ID}/resolve/{REVISION}/{FILENAME}")
        print(f"没有 huggingface_hub，用 urllib 下载：{url}")
        import urllib.request
        req = urllib.request.Request(url, headers={"User-Agent": "trlab-fetch-bf16"})
        with urllib.request.urlopen(req, timeout=60) as r, open(PART, "wb") as f:
            done = 0
            while True:
                buf = r.read(8 << 20)
                if not buf:
                    break
                f.write(buf)
                done += len(buf)
                if done % (1 << 30) < (8 << 20):
                    print(f"  ... 已下载 {human(done)}（{time.time() - t0:.0f}s）", flush=True)
        os.replace(PART, DST)

size = DST.stat().st_size
print(f"\n字节数：{size}（期望 {EXPECT_BYTES}）→ {'一致' if size == EXPECT_BYTES else '不一致！'}")
assert size == EXPECT_BYTES, (
    f"字节数不符：{size} != {EXPECT_BYTES}。revision 可能被上游替换，或下载不完整。")

digest = sha256_of(DST)
print(f"sha256：{digest}")
print(f"期望  ：{EXPECT_SHA256}")
assert digest == EXPECT_SHA256, (
    "sha256 不符 —— 这不是 pinned revision 的那个文件，拒绝使用。"
    "主 notebook 的 attached_input 分支会做同样的强校验。")

print(f"\n完成（{time.time() - t0:.0f}s）。下一步：Save Version（Quick Save）→ "
      f"回主 notebook → Add Input → 选本 notebook 的输出。")
print(f"挂载后路径形如 /kaggle/input/<slug>/{FILENAME}，主 notebook 会自动找到它。")


## 故障排查

| 现象 | 处置 |
|---|---|
| `ModuleNotFoundError: huggingface_hub` | 代码会自动回落到 `urllib`，不用管 |
| `HTTPError 401 / 404` | revision 或文件名写错；本文件的两个值都与主 notebook 的 `S4_PINNED` 一致，不要改 |
| 下载到一半断 | 重跑本 cell。`hf_hub_download` 自己有缓存/续传；`urllib` 路径会留下 `.partial`，删掉重下即可 |
| `字节数不符` / `sha256 不符` | 上游文件被替换或网络返回错误页面。**不要手动放行**，主 notebook 也会拒绝 |
| working 空间不够 | 20 GB 配额下 15.26 GiB 单文件可行，但别同时放 Q8_0（8.11 GiB）—— 两个文件要分两个 notebook |

## 这个 notebook 不做什么

- 不转格式、不量化、不跑推理；
- 不装任何包；
- 不引用主 notebook 的任何状态 —— 它可以独立存在、独立重跑。
